In [ ]:
#@title Gemini Live Transcriber { display-mode: "form" }
# One run. Prompts: model -> API key -> upload -> automatic transcription.

import os, sys, re, math, time, json, shutil, asyncio, subprocess, getpass, zipfile
from array import array
from pathlib import Path
from datetime import datetime
from google.colab import files, userdata

SAMPLE_RATE = 16_000
BYTES_PER_SAMPLE = 2
BYTES_PER_SEC = SAMPLE_RATE * BYTES_PER_SAMPLE
FRAME_MS = 100
FRAME_BYTES = BYTES_PER_SEC * FRAME_MS // 1000
OVERLAP_SEC = 1.0
STALL_ACTIVE_AUDIO_SEC = 12.0
STALL_RMS_THRESHOLD = 160
FINAL_IDLE_SEC = 6.0
TMP = Path('/content/gemini_live_tmp')
TMP.mkdir(parents=True, exist_ok=True)

MODELS = {
    '1': {
        'name': 'Gemini 3.5 Transcribe Live',
        'id': 'gemini-3.5-transcribe-live',
        'tpm': 20_000,
        'concurrency': 2,
        'max_chunk_sec': 540,
    },
    '2': {
        'name': 'Gemini 3.8 Live',
        'id': 'gemini-3.8-live',
        'tpm': 65_000,
        'concurrency': 40,
        'max_chunk_sec': 480,
    },
}


def choose_models():
    print('Choose model:')
    print('  1 = Gemini 3.5 Transcribe Live')
    print('  2 = Gemini 3.8 Live')
    print('  3 = Both (3.5 first, then 3.8)')
    while True:
        choice = input('Model [1]: ').strip() or '1'
        if choice in ('1', '2', '3'):
            return ['1', '2'] if choice == '3' else [choice]
        print('Type 1, 2, or 3.')


def ask_key():
    try:
        key = (userdata.get('GeminiAPIKey') or '').strip()
    except Exception as e:
        raise RuntimeError(
            "Colab secret 'GeminiAPIKey' is missing or not enabled for this notebook."
        ) from e
    if not key:
        raise RuntimeError(
            "Colab secret 'GeminiAPIKey' is empty or not enabled for this notebook."
        )
    print("[secret] GeminiAPIKey loaded from Colab Secrets.")
    return key


def upload_one_file():
    print('\nChoose the recording to upload...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file was uploaded.')
    if len(uploaded) != 1:
        raise RuntimeError('Please upload exactly one recording.')
    name, data = next(iter(uploaded.items()))
    path = Path('/content') / Path(name).name
    path.write_bytes(data)
    return path


def install_sdk():
    print('\n[setup] Preparing Gemini SDK...', flush=True)
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'google-genai==2.25.0'],
        check=True,
    )
    global genai, types
    from google import genai as _genai
    from google.genai import types as _types
    genai, types = _genai, _types


def ffmpeg_to_pcm(src, dst):
    if not shutil.which('ffmpeg'):
        raise RuntimeError('ffmpeg is unavailable in this Colab runtime.')
    subprocess.run([
        'ffmpeg', '-hide_banner', '-loglevel', 'error', '-y',
        '-i', str(src), '-vn', '-ac', '1', '-ar', str(SAMPLE_RATE),
        '-f', 's16le', str(dst),
    ], check=True)
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError('Audio conversion failed or produced an empty file.')
    return dst.stat().st_size / BYTES_PER_SEC


def hms(sec):
    sec = max(0, int(round(sec)))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    return f'{h:02d}:{m:02d}:{s:02d}'


def safe_name(s):
    return re.sub(r'[^\w.-]+', '_', s, flags=re.UNICODE).strip('_') or 'transcript'


def make_chunks(total_sec, spec):
    c = min(spec['concurrency'], max(1, math.ceil(total_sec / 2)))
    waves = max(1, math.ceil(total_sec / (c * spec['max_chunk_sec'])))
    count = min(max(1, waves * c), max(1, math.ceil(total_sec / 2)))
    chunk_len = total_sec / count
    out = []
    for i in range(count):
        nominal_start = i * chunk_len
        nominal_end = total_sec if i == count - 1 else (i + 1) * chunk_len
        out.append({
            'index': i,
            'start_sec': max(0.0, nominal_start - (OVERLAP_SEC if i else 0.0)),
            'end_sec': min(total_sec, nominal_end + (OVERLAP_SEC if i < count - 1 else 0.0)),
        })
    return out


def norm_word(w):
    return re.sub(r'[^\w\u0600-\u06FF]+', '', w, flags=re.UNICODE).casefold()


def merge_overlap(parts, max_words=60):
    merged = []
    for text in parts:
        words = text.strip().split()
        if not words:
            continue
        if not merged:
            merged = words
            continue
        left = [norm_word(x) for x in merged]
        right = [norm_word(x) for x in words]
        best = 0
        for n in range(min(max_words, len(merged), len(words)), 1, -1):
            if left[-n:] == right[:n] and any(left[-n:]):
                best = n
                break
        merged.extend(words[best:])
    return ' '.join(merged).strip()


def pcm_rms(data):
    if len(data) < 2:
        return 0.0
    samples = array('h')
    samples.frombytes(data[:len(data) - (len(data) % 2)])
    if sys.byteorder != 'little':
        samples.byteswap()
    if not samples:
        return 0.0
    return math.sqrt(sum(v * v for v in samples) / len(samples))


def probably_silent(path, start_byte, end_byte, threshold=120):
    span = max(0, end_byte - start_byte)
    if span <= 0:
        return True
    window = min(BYTES_PER_SEC // 2, span)
    positions = [start_byte] if span <= window else [
        start_byte + int((span - window) * i / 7) for i in range(8)
    ]
    peak = 0.0
    with open(path, 'rb') as f:
        for pos in positions:
            pos -= pos % 2
            f.seek(pos)
            data = f.read(window)
            if len(data) < 2:
                continue
            samples = array('h')
            samples.frombytes(data[:len(data) - (len(data) % 2)])
            if sys.byteorder != 'little':
                samples.byteswap()
            if samples:
                rms = math.sqrt(sum(v * v for v in samples) / len(samples))
                peak = max(peak, rms)
    return peak < threshold


def model_config(key):
    if key == '1':
        return {
            'response_modalities': ['TEXT'],
            'realtime_input_config': {
                'automatic_activity_detection': {'disabled': True},
            },
            'input_audio_transcription': {
                'language_codes': [],
                'mode': 'VERBATIM',
            },
        }
    return {
        'response_modalities': ['AUDIO'],
        'input_audio_transcription': {},
    }


async def receive_transcript(session, stream_done, telemetry):
    pieces = []
    iterator = session.receive().__aiter__()
    while True:
        timeout = 30.0 if not stream_done.is_set() else (FINAL_IDLE_SEC if pieces else 20.0)
        try:
            msg = await asyncio.wait_for(iterator.__anext__(), timeout=timeout)
        except StopAsyncIteration:
            telemetry['receive_end'] += 1
            break
        except asyncio.TimeoutError:
            telemetry['receive_timeout'] += 1
            break
        except Exception as e:
            code = getattr(e, 'code', None)
            if code == 1000 or str(e).lstrip().startswith('1000'):
                telemetry['normal_close'] += 1
                break
            raise

        telemetry['messages'] += 1
        telemetry['last_server_at'] = time.monotonic()
        sc = getattr(msg, 'server_content', None)
        if not sc:
            telemetry['other_messages'] += 1
            continue

        interim_tr = getattr(sc, 'interim_input_transcription', None)
        if interim_tr and (getattr(interim_tr, 'text', '') or '').strip():
            telemetry['interim_events'] += 1

        tr = getattr(sc, 'input_transcription', None)
        text = (getattr(tr, 'text', '') or '').strip() if tr else ''
        if text:
            telemetry['final_events'] += 1
            if not pieces or text != pieces[-1]:
                pieces.append(text)

    return ' '.join(pieces).strip()


async def transcribe_chunk(client, pcm_path, chunk, model_key, max_attempts=3):
    spec = MODELS[model_key]
    start_byte = int(chunk['start_sec'] * BYTES_PER_SEC)
    end_byte = int(chunk['end_sec'] * BYTES_PER_SEC)
    start_byte -= start_byte % 2
    end_byte -= end_byte % 2
    total = max(0, end_byte - start_byte)
    last_error = None

    for attempt in range(1, max_attempts + 1):
        recv = None
        done = asyncio.Event()
        telemetry = {
            'messages': 0,
            'last_server_at': None,
            'interim_events': 0,
            'final_events': 0,
            'receive_timeout': 0,
            'receive_end': 0,
            'normal_close': 0,
            'other_messages': 0,
        }
        active_without_server = 0.0
        seen_messages = 0
        try:
            async with client.aio.live.connect(model=spec['id'], config=model_config(model_key)) as session:
                recv = asyncio.create_task(receive_transcript(session, done, telemetry))
                if model_key == '1':
                    await session.send_realtime_input(activity_start=types.ActivityStart())
                remaining = total
                with open(pcm_path, 'rb', buffering=1024 * 1024) as fh:
                    fh.seek(start_byte)
                    next_send = time.monotonic()
                    while remaining > 0:
                        data = fh.read(min(FRAME_BYTES, remaining))
                        if not data:
                            break

                        await session.send_realtime_input(
                            audio=types.Blob(data=data, mime_type='audio/pcm;rate=16000')
                        )
                        remaining -= len(data)

                        if telemetry['messages'] != seen_messages:
                            seen_messages = telemetry['messages']
                            active_without_server = 0.0
                        elif model_key == '1' and pcm_rms(data) >= STALL_RMS_THRESHOLD:
                            active_without_server += len(data) / BYTES_PER_SEC
                            if active_without_server >= STALL_ACTIVE_AUDIO_SEC:
                                raise RuntimeError(
                                    'silent Live session: no server responses during '
                                    f'{active_without_server:.1f}s of active audio'
                                )

                        next_send += FRAME_MS / 1000.0
                        delay = next_send - time.monotonic()
                        if delay > 0:
                            await asyncio.sleep(delay)

                if model_key == '1':
                    await session.send_realtime_input(activity_end=types.ActivityEnd())
                else:
                    await session.send_realtime_input(audio_stream_end=True)
                done.set()
                try:
                    text = await asyncio.wait_for(recv, timeout=30.0)
                finally:
                    if recv is not None and not recv.done():
                        recv.cancel()

                if not text and not await asyncio.to_thread(
                    probably_silent, pcm_path, start_byte, end_byte
                ):
                    raise RuntimeError(
                        'empty transcript on non-silent audio; '
                        f'server_messages={telemetry["messages"]}, '
                        f'interim={telemetry["interim_events"]}, '
                        f'final={telemetry["final_events"]}'
                    )
                return text
        except Exception as e:
            last_error = e
            done.set()
            if recv is not None and not recv.done():
                recv.cancel()
                try:
                    await recv
                except BaseException:
                    pass
            if attempt < max_attempts:
                await asyncio.sleep(2 * attempt)

    raise RuntimeError(f'chunk {chunk["index"] + 1} failed: {last_error}')


async def transcribe_model(api_key, pcm_path, duration, model_key):
    spec = MODELS[model_key]
    chunks = make_chunks(duration, spec)
    concurrency = min(spec['concurrency'], len(chunks))
    results = [None] * len(chunks)
    client = genai.Client(api_key=api_key)
    started = time.monotonic()
    done_count = 0
    stop_heartbeat = asyncio.Event()

    expected = max(ch['end_sec'] - ch['start_sec'] for ch in chunks) * math.ceil(len(chunks) / concurrency)
    print(
        f'\n[{spec["name"]}] {len(chunks)} chunks | up to {concurrency} parallel | '
        f'audio {hms(duration)} | rough minimum ~{expected/60:.1f} min',
        flush=True,
    )

    async def heartbeat():
        while not stop_heartbeat.is_set():
            elapsed = time.monotonic() - started
            print(
                f'\r[{spec["name"]}] working... {done_count}/{len(chunks)} chunks | elapsed {hms(elapsed)}',
                end='', flush=True,
            )
            try:
                await asyncio.wait_for(stop_heartbeat.wait(), timeout=10)
            except asyncio.TimeoutError:
                pass

    hb = asyncio.create_task(heartbeat())

    async def worker(ch):
        nonlocal done_count
        text = await transcribe_chunk(client, pcm_path, ch, model_key)
        results[ch['index']] = text
        done_count += 1

    async def run_wave(wave):
        tasks = [asyncio.create_task(worker(ch)) for ch in wave]
        settled = await asyncio.gather(*tasks, return_exceptions=True)
        return [x for x in settled if isinstance(x, Exception)]

    try:
        failures = []
        for offset in range(0, len(chunks), concurrency):
            failures.extend(await run_wave(chunks[offset:offset + concurrency]))

        missing = [ch for ch in chunks if results[ch['index']] is None]
        recovery = max(1, concurrency // 2)
        round_no = 0
        while missing and round_no < 5:
            round_no += 1
            print(f'\n[{spec["name"]}] retrying {len(missing)} failed chunk(s) with concurrency {recovery}...', flush=True)
            for offset in range(0, len(missing), recovery):
                await run_wave(missing[offset:offset + recovery])
            missing = [ch for ch in chunks if results[ch['index']] is None]
            recovery = max(1, recovery // 2)

        if missing:
            raise RuntimeError(
                f'{len(missing)} chunk(s) still failed after retries; no incomplete transcript was saved.'
            )

        transcript = merge_overlap(results)
        elapsed = time.monotonic() - started
        print(f'\r[{spec["name"]}] done: {len(chunks)}/{len(chunks)} chunks | {elapsed/60:.2f} min' + ' ' * 20, flush=True)
        return transcript, elapsed
    finally:
        stop_heartbeat.set()
        try:
            await hb
        except Exception:
            pass
        try:
            client.close()
        except Exception:
            pass


async def main():
    chosen = choose_models()
    api_key = ask_key()
    src = upload_one_file()
    install_sdk()

    pcm = TMP / f'{int(time.time())}_{safe_name(src.stem)}.pcm'
    print('\n[1/3] Preparing audio...', flush=True)
    prep_start = time.monotonic()
    duration = await asyncio.to_thread(ffmpeg_to_pcm, src, pcm)
    print(f'[1/3] Ready: {src.name} | duration {hms(duration)} | prep {time.monotonic()-prep_start:.1f}s', flush=True)

    stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = safe_name(src.stem)
    outputs = []
    timings = []

    try:
        print('[2/3] Transcribing...', flush=True)
        for key in chosen:
            text, elapsed = await transcribe_model(api_key, pcm, duration, key)
            spec = MODELS[key]
            out = Path('/content') / f'{base}_{spec["id"]}_{stamp}.txt'
            out.write_text(text, encoding='utf-8')
            outputs.append(out)
            timings.append((spec['name'], elapsed))

        print('\n[3/3] Finished.', flush=True)
        for name, elapsed in timings:
            print(f'  {name}: {elapsed/60:.2f} min')

        if len(outputs) == 1:
            print(f'\nDownloading: {outputs[0].name}', flush=True)
            files.download(str(outputs[0]))
        else:
            z = Path('/content') / f'{base}_Gemini_Live_transcripts_{stamp}.zip'
            with zipfile.ZipFile(z, 'w', zipfile.ZIP_DEFLATED) as archive:
                for p in outputs:
                    archive.write(p, arcname=p.name)
            print(f'\nDownloading: {z.name}', flush=True)
            files.download(str(z))
    finally:
        try:
            pcm.unlink(missing_ok=True)
        except Exception:
            pass


await main()


In [ ]:
#@title Gemini 3.5 Live - Chunk Integrity Benchmark { display-mode: "form" }
# Fully automatic: Colab Secret -> GitHub test audio -> correctness reference -> chunk-size benchmark -> c=6 estimate.

import sys, math, time, json, asyncio, subprocess, shutil, urllib.request, re, difflib
from array import array
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
from google.colab import files, userdata

MODEL_ID = "gemini-3.5-transcribe-live"
SDK_VERSION = "2.25.0"
SECRET_NAME = "GeminiAPIKey"
LANGUAGE_CODES = ["ar-EG", "en-US"]

TEST_AUDIO_NAME = "62_1790064377959_Food poisoning_compressed.ogg"
TEST_AUDIO_URL = "https://raw.githubusercontent.com/abdullahsamirashour/gpt/main/gemini-live-transcriber/62_1790064377959_Food%20poisoning_compressed.ogg"

SAMPLE_RATE = 16_000
BYTES_PER_SEC = SAMPLE_RATE * 2
FRAME_MS = 100
FRAME_BYTES = BYTES_PER_SEC * FRAME_MS // 1000

CONCURRENCY = 6
REGION_COUNT = 6
REFERENCE_UNIT_SEC = 10
REFERENCE_SPAN_SEC = 60
CANDIDATE_CHUNK_SECS = (60, 40, 30, 20)
FINAL_GRACE_SEC = 6.0
MAX_EMPTY_RETRIES = 1

# Strict enough to reject obviously truncated output while tolerating normal ASR variation.
MIN_LENGTH_RATIO = 0.60
MIN_REFERENCE_RECALL = 0.55
MIN_SCORABLE_REFERENCE_WORDS = 5
MIN_SCORABLE_REGIONS = 4

TMP = Path("/content/gemini35_chunk_bench")
TMP.mkdir(parents=True, exist_ok=True)


def load_key():
    try:
        key = (userdata.get(SECRET_NAME) or "").strip()
    except Exception as e:
        raise RuntimeError(
            f"Colab Secret '{SECRET_NAME}' is missing or not enabled for this notebook."
        ) from e
    if not key:
        raise RuntimeError(
            f"Colab Secret '{SECRET_NAME}' is empty or not enabled for this notebook."
        )
    print(f"[secret] {SECRET_NAME} loaded from Colab Secrets.")
    return key


def download_test_audio():
    path = Path("/content") / TEST_AUDIO_NAME
    print(f"[test audio] Downloading {TEST_AUDIO_NAME} from GitHub...", flush=True)
    urllib.request.urlretrieve(TEST_AUDIO_URL, path)
    if not path.exists() or path.stat().st_size == 0:
        raise RuntimeError("Downloaded benchmark recording is empty.")
    print(f"[test audio] Ready: {path.stat().st_size / 1024 / 1024:.2f} MiB")
    return path


def install_sdk():
    print(f"[setup] google-genai=={SDK_VERSION}", flush=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "--upgrade", f"google-genai=={SDK_VERSION}"],
        check=True,
    )
    global genai, types, installed_sdk
    from google import genai as _genai
    from google.genai import types as _types
    import importlib.metadata as _metadata
    genai, types = _genai, _types
    installed_sdk = _metadata.version("google-genai")


def ffmpeg_to_pcm(src, dst):
    if not shutil.which("ffmpeg"):
        raise RuntimeError("ffmpeg is unavailable in this Colab runtime.")
    subprocess.run(
        [
            "ffmpeg", "-hide_banner", "-loglevel", "error", "-y",
            "-i", str(src), "-vn", "-ac", "1", "-ar", str(SAMPLE_RATE),
            "-f", "s16le", str(dst),
        ],
        check=True,
    )
    if not dst.exists() or dst.stat().st_size == 0:
        raise RuntimeError("Audio conversion failed.")
    return dst.stat().st_size / BYTES_PER_SEC


def manual_config():
    return types.LiveConnectConfig(
        response_modalities=["TEXT"],
        realtime_input_config=types.RealtimeInputConfig(
            automatic_activity_detection=types.AutomaticActivityDetection(disabled=True)
        ),
        input_audio_transcription=types.AudioTranscriptionConfig(
            language_codes=LANGUAGE_CODES,
            mode="VERBATIM",
        ),
    )


def pcm_rms(path, start_sec, seconds):
    start = int(start_sec * BYTES_PER_SEC) // 2 * 2
    size = int(seconds * BYTES_PER_SEC) // 2 * 2
    with open(path, "rb") as fh:
        fh.seek(start)
        data = fh.read(size)
    samples = array("h")
    samples.frombytes(data[: len(data) - (len(data) % 2)])
    if sys.byteorder != "little":
        samples.byteswap()
    if not samples:
        return 0.0
    return math.sqrt(sum(v * v for v in samples) / len(samples))


def select_regions(pcm_path, duration):
    span = min(float(REFERENCE_SPAN_SEC), duration)
    if duration <= span:
        return [0.0]

    # Score a grid, then greedily keep speech-heavy, non-overlapping windows.
    max_start = duration - span
    grid = [max_start * i / 23 for i in range(24)]
    scored = sorted(
        ((pcm_rms(pcm_path, s, min(12.0, span)), s) for s in grid),
        reverse=True,
    )

    chosen = []
    for _rms, start in scored:
        if all(abs(start - existing) >= span for existing in chosen):
            chosen.append(start)
        if len(chosen) == REGION_COUNT:
            break

    # If energy-based spacing did not produce enough, fill with evenly spaced windows.
    if len(chosen) < REGION_COUNT:
        for i in range(REGION_COUNT):
            start = max_start * i / max(1, REGION_COUNT - 1)
            if all(abs(start - existing) >= span * 0.8 for existing in chosen):
                chosen.append(start)
            if len(chosen) == REGION_COUNT:
                break

    return sorted(chosen[:REGION_COUNT])


def norm_word(word):
    return re.sub(r"[^\w\u0600-\u06FF]+", "", word, flags=re.UNICODE).casefold()


def merge_overlap(parts, max_words=50):
    merged = []
    for text in parts:
        words = (text or "").strip().split()
        if not words:
            continue
        if not merged:
            merged = words
            continue
        left = [norm_word(x) for x in merged]
        right = [norm_word(x) for x in words]
        best = 0
        for n in range(min(max_words, len(merged), len(words)), 1, -1):
            if left[-n:] == right[:n] and any(left[-n:]):
                best = n
                break
        merged.extend(words[best:])
    return " ".join(merged).strip()


def normalize_tokens(text):
    return [norm_word(w) for w in (text or "").split() if norm_word(w)]


def compare_text(reference, candidate):
    ref = normalize_tokens(reference)
    cand = normalize_tokens(candidate)

    if len(ref) < MIN_SCORABLE_REFERENCE_WORDS:
        return {
            "reference_words": len(ref),
            "candidate_words": len(cand),
            "length_ratio": None,
            "reference_recall": None,
            "sequence_ratio": None,
            "scorable": False,
            "pass": None,
        }

    if not cand:
        return {
            "reference_words": len(ref),
            "candidate_words": 0,
            "length_ratio": 0.0,
            "reference_recall": 0.0,
            "sequence_ratio": 0.0,
            "scorable": True,
            "pass": False,
        }

    ref_counts = Counter(ref)
    cand_counts = Counter(cand)
    overlap = sum((ref_counts & cand_counts).values())
    recall = overlap / len(ref)
    length_ratio = len(cand) / len(ref)
    seq = difflib.SequenceMatcher(None, ref, cand).ratio()

    # We care about missing reference speech. Extra candidate words are not
    # penalized here because the 10s reference can itself miss speech.
    passed = (
        length_ratio >= MIN_LENGTH_RATIO
        and recall >= MIN_REFERENCE_RECALL
    )

    return {
        "reference_words": len(ref),
        "candidate_words": len(cand),
        "length_ratio": round(length_ratio, 3),
        "reference_recall": round(recall, 3),
        "sequence_ratio": round(seq, 3),
        "scorable": True,
        "pass": passed,
    }


async def stream_audio(session, pcm_path, start_sec, seconds):
    start = int(start_sec * BYTES_PER_SEC) // 2 * 2
    remaining = int(seconds * BYTES_PER_SEC) // 2 * 2

    await session.send_realtime_input(activity_start=types.ActivityStart())

    with open(pcm_path, "rb", buffering=1024 * 1024) as fh:
        fh.seek(start)
        next_send = time.monotonic()
        while remaining > 0:
            data = fh.read(min(FRAME_BYTES, remaining))
            if not data:
                break
            await session.send_realtime_input(
                audio=types.Blob(data=data, mime_type="audio/pcm;rate=16000")
            )
            remaining -= len(data)
            next_send += FRAME_MS / 1000.0
            delay = next_send - time.monotonic()
            if delay > 0:
                await asyncio.sleep(delay)

    await session.send_realtime_input(activity_end=types.ActivityEnd())


async def receive_segment(session, state):
    try:
        async for response in session.receive():
            state["messages"] += 1
            sc = getattr(response, "server_content", None)
            if not sc:
                continue
            state["server_content_messages"] += 1

            interim = getattr(sc, "interim_input_transcription", None)
            if interim and (getattr(interim, "text", "") or "").strip():
                state["interim_events"] += 1
                state["latest_interim"] = interim.text.strip()

            final = getattr(sc, "input_transcription", None)
            if final and (getattr(final, "text", "") or "").strip():
                state["final_events"] += 1
                state["final_segments"].append(final.text.strip())

    except asyncio.CancelledError:
        raise
    except Exception as e:
        code = getattr(e, "code", None)
        if code != 1000 and not str(e).lstrip().startswith("1000"):
            state["receiver_error"] = f"{type(e).__name__}: {e}"


async def one_segment(client, pcm_path, start_sec, seconds, label, attempt=1):
    state = {
        "label": label,
        "start_sec": round(start_sec, 3),
        "audio_sec": round(seconds, 3),
        "attempt": attempt,
        "messages": 0,
        "server_content_messages": 0,
        "interim_events": 0,
        "final_events": 0,
        "final_segments": [],
        "latest_interim": "",
        "receiver_error": "",
        "api_error": "",
    }
    started = time.monotonic()
    recv = None

    try:
        async with client.aio.live.connect(model=MODEL_ID, config=manual_config()) as session:
            recv = asyncio.create_task(receive_segment(session, state))
            await stream_audio(session, pcm_path, start_sec, seconds)

            # A final commonly arrives a few seconds after activity_end.
            # If it never arrives but interim text exists, keep the last interim as a measured fallback.
            deadline = time.monotonic() + FINAL_GRACE_SEC
            while time.monotonic() < deadline:
                if state["final_events"]:
                    await asyncio.sleep(1.0)
                    break
                await asyncio.sleep(0.1)

    except Exception as e:
        state["api_error"] = f"{type(e).__name__}: {e}"
    finally:
        if recv is not None and not recv.done():
            recv.cancel()
            try:
                await recv
            except BaseException:
                pass

    final_text = merge_overlap(state["final_segments"])
    interim_text = state["latest_interim"].strip()

    # Final is authoritative, but on this recording we have observed a short
    # final followed by a materially longer interim hypothesis. For benchmark
    # coverage, use the more complete single hypothesis instead of concatenating
    # both and double-counting overlapping speech.
    if final_text and interim_text:
        if len(normalize_tokens(interim_text)) > len(normalize_tokens(final_text)):
            effective = interim_text
            mode = "interim-preferred"
        else:
            effective = final_text
            mode = "final-preferred"
    elif final_text:
        effective = final_text
        mode = "final"
    elif interim_text:
        effective = interim_text
        mode = "interim-fallback"
    else:
        effective = ""
        mode = "empty"

    state["effective_text"] = effective
    state["mode"] = mode
    state["ok"] = bool(effective) and not state["api_error"]
    state["elapsed_sec"] = round(time.monotonic() - started, 3)
    state["effective_words"] = len(normalize_tokens(effective))
    return state


async def segment_with_retry(client, pcm_path, start_sec, seconds, label):
    result = await one_segment(client, pcm_path, start_sec, seconds, label, 1)
    for attempt in range(2, MAX_EMPTY_RETRIES + 2):
        if result["ok"]:
            break
        await asyncio.sleep(1)
        result = await one_segment(client, pcm_path, start_sec, seconds, label, attempt)
    return result


async def run_wave(client, pcm_path, jobs):
    started = time.monotonic()
    results = await asyncio.gather(*[
        segment_with_retry(client, pcm_path, start, seconds, label)
        for start, seconds, label in jobs
    ])
    return results, time.monotonic() - started


def mode_counts(results):
    return dict(Counter(r["mode"] for r in results))


async def build_reference(client, pcm_path, regions):
    jobs = []
    units_per_region = REFERENCE_SPAN_SEC // REFERENCE_UNIT_SEC
    for region_i, region_start in enumerate(regions):
        for unit_i in range(units_per_region):
            start = region_start + unit_i * REFERENCE_UNIT_SEC
            jobs.append((
                start,
                REFERENCE_UNIT_SEC,
                f"reference-r{region_i+1}-u{unit_i+1}",
            ))

    print(
        f"\n[1] Building reference: {len(jobs)} independent "
        f"{REFERENCE_UNIT_SEC}s sessions in waves of {CONCURRENCY}"
    )

    all_results = []
    for offset in range(0, len(jobs), CONCURRENCY):
        wave_jobs = jobs[offset: offset + CONCURRENCY]
        results, elapsed = await run_wave(client, pcm_path, wave_jobs)
        all_results.extend(results)
        print(
            f"  wave {offset//CONCURRENCY + 1}: "
            f"{sum(r['ok'] for r in results)}/{len(results)} usable | "
            f"{elapsed:.1f}s | modes={mode_counts(results)}"
        )

    by_region = {}
    for region_i, region_start in enumerate(regions):
        region_results = [
            r for r in all_results
            if r["label"].startswith(f"reference-r{region_i+1}-")
        ]
        by_region[region_i] = {
            "start_sec": region_start,
            "results": region_results,
            "text": " ".join(r["effective_text"] for r in region_results if r["ok"]).strip(),
        }

    bad = [r for r in all_results if not r["ok"]]
    scorable_regions = sum(
        len(normalize_tokens(region["text"])) >= MIN_SCORABLE_REFERENCE_WORDS
        for region in by_region.values()
    )
    return {
        "ok": scorable_regions >= MIN_SCORABLE_REGIONS,
        "results": all_results,
        "regions": by_region,
        "bad_count": len(bad),
        "scorable_regions": scorable_regions,
    }


def reference_prefix(reference_region, seconds):
    units = max(1, int(math.ceil(seconds / REFERENCE_UNIT_SEC)))
    return " ".join(
        r["effective_text"]
        for r in reference_region["results"][:units]
        if r["ok"]
    ).strip()


async def test_candidate_size(client, pcm_path, reference, seconds):
    jobs = [
        (
            region["start_sec"],
            seconds,
            f"candidate-{seconds}s-r{region_i+1}",
        )
        for region_i, region in reference["regions"].items()
    ]

    results, elapsed = await run_wave(client, pcm_path, jobs)

    comparisons = []
    for region_i, result in enumerate(results):
        ref_text = reference_prefix(reference["regions"][region_i], seconds)
        metrics = compare_text(ref_text, result["effective_text"])
        comparisons.append(metrics)
        result["comparison"] = metrics

    scorable = [
        (r, c) for r, c in zip(results, comparisons) if c["scorable"]
    ]
    passed = sum(r["ok"] and c["pass"] for r, c in scorable)
    return {
        "seconds": seconds,
        "results": results,
        "comparisons": comparisons,
        "passed": passed,
        "total": len(scorable),
        "jobs": len(results),
        "wave_elapsed_sec": round(elapsed, 3),
        "modes": mode_counts(results),
    }


async def main():
    report = {
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": MODEL_ID,
        "sdk_target_version": SDK_VERSION,
        "concurrency": CONCURRENCY,
        "candidate_chunk_secs": list(CANDIDATE_CHUNK_SECS),
    }

    print("Gemini 3.5 Transcribe Live - chunk integrity benchmark")
    print(f"No prompts: Colab Secret + GitHub audio | language hints={LANGUAGE_CODES}.")

    api_key = load_key()
    src = download_test_audio()
    install_sdk()

    pcm = TMP / "test_audio.pcm"
    client = None

    try:
        duration = await asyncio.to_thread(ffmpeg_to_pcm, src, pcm)
        report["source_duration_sec"] = duration
        report["google_genai_version"] = installed_sdk
        regions = select_regions(pcm, duration)
        report["regions"] = regions

        print(
            f"\n[0] Audio: {duration/60:.2f} min | "
            f"selected region starts: {', '.join(f'{x:.1f}s' for x in regions)}"
        )

        client = genai.Client(api_key=api_key)

        reference = await build_reference(client, pcm, regions)
        report["reference"] = reference

        if not reference["ok"]:
            print(
                f"\nSTOP: only {reference['scorable_regions']} regions had enough "
                "reference speech for a meaningful completeness test."
            )
            report["validated_chunk_sec"] = None
            return report

        if reference["bad_count"]:
            print(
                f"  note: {reference['bad_count']} 10s reference unit(s) were empty "
                "after retry; they are treated as unscorable, not as benchmark failure."
            )

        fallback_count = sum(
            r["mode"] == "interim-fallback" for r in reference["results"]
        )
        print(
            f"  reference complete: {len(reference['results'])} units | "
            f"interim fallbacks={fallback_count}"
        )

        print("\n[2] Candidate chunk-size completeness at c=6")
        print("  size | coverage | wave time | modes | median length | median ref-recall")

        candidate_reports = []
        selected = None

        for seconds in CANDIDATE_CHUNK_SECS:
            candidate = await test_candidate_size(client, pcm, reference, seconds)
            candidate_reports.append(candidate)

            scored = [c for c in candidate["comparisons"] if c["scorable"]]
            lengths = sorted(c["length_ratio"] for c in scored)
            recalls = sorted(c["reference_recall"] for c in scored)
            mid = len(lengths) // 2
            med_len = lengths[mid] if lengths else 0.0
            med_recall = recalls[mid] if recalls else 0.0

            print(
                f"  {seconds:>4}s | {candidate['passed']}/{candidate['total']}     | "
                f"{candidate['wave_elapsed_sec']:>8.1f}s | {candidate['modes']} | "
                f"{med_len:.2f} | {med_recall:.2f}"
            )

            for result in candidate["results"]:
                c = result["comparison"]
                if not c["scorable"]:
                    print(
                        f"       SKIP {result['label']}: reference has only "
                        f"{c['reference_words']} word(s)"
                    )
                elif not (result["ok"] and c["pass"]):
                    print(
                        f"       FAIL {result['label']}: mode={result['mode']} "
                        f"len={c['length_ratio']:.2f} recall={c['reference_recall']:.2f} "
                        f"seq={c['sequence_ratio']:.2f} "
                        f"api={result['api_error'][:80]}"
                    )

            if (
                candidate["total"] >= MIN_SCORABLE_REGIONS
                and candidate["passed"] == candidate["total"]
                and selected is None
            ):
                selected = candidate
                # Candidate sizes are longest -> shortest, so first full pass wins.
                break

        report["candidates"] = candidate_reports

        if selected is None:
            selected_sec = REFERENCE_UNIT_SEC
            # Use measured reference wave time as the conservative fallback estimate.
            ref_elapsed = max(r["elapsed_sec"] for r in reference["results"][:CONCURRENCY])
            selected_wave_elapsed = ref_elapsed
            print(
                f"\nRESULT: no larger candidate fully matched the reference. "
                f"Use {selected_sec}s independent sessions."
            )
        else:
            selected_sec = selected["seconds"]
            selected_wave_elapsed = selected["wave_elapsed_sec"]
            print(f"\nRESULT: validated chunk size = {selected_sec}s at concurrency {CONCURRENCY}")

        chunks = math.ceil(duration / selected_sec)
        waves = math.ceil(chunks / CONCURRENCY)
        estimate_min = waves * selected_wave_elapsed / 60.0
        ideal_floor_min = duration / CONCURRENCY / 60.0

        report["validated_chunk_sec"] = selected_sec
        report["production_estimate"] = {
            "chunks": chunks,
            "waves": waves,
            "measured_wave_sec": round(selected_wave_elapsed, 3),
            "estimated_wall_clock_min": round(estimate_min, 3),
            "ideal_stream_floor_min": round(ideal_floor_min, 3),
        }

        print(
            f"  full recording: {chunks} chunks / {waves} waves | "
            f"measured estimate ~{estimate_min:.2f} min | "
            f"ideal audio floor {ideal_floor_min:.2f} min"
        )
        return report

    finally:
        if client is not None:
            try:
                client.close()
            except Exception:
                pass
        try:
            pcm.unlink(missing_ok=True)
        except Exception:
            pass


report = await main()
report_path = Path("/content/gemini35_live_chunk_integrity_report.json")
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"\nSaved report: {report_path}")
files.download(str(report_path))
